# PCM-Tiny Structured Pruning Experiment
**3D Mamba: Is This Really Efficient?**

## 환경 스펙
- GPU: NVIDIA L4 (Colab Pro)
- causal-conv1d: 1.0.0 (PCM 자체 소스)
- mamba-ssm: 1.0.1 (PCM 자체 소스)
- transformers: 4.38.2
- 데이터: ScanObjectNN PB-T50-RS
- 모델: PCM-Tiny (6.94M)

## 실행 순서
**런타임 시작 시마다**: Step 1~6을 순서대로 실행
**데이터가 없을 때만**: Step 7 실행
**학습**: Step 8 (동작 확인) → Step 9 (본 학습)

## Step 1. PCM Repo 클론
런타임 시작 시마다 실행 (런타임 재시작하면 /content가 초기화됨)

In [ ]:
%cd /content
!git clone https://github.com/SkyworkAI/PointCloudMamba.git
%cd /content/PointCloudMamba
print('✅ 클론 완료')

/content
Cloning into 'PointCloudMamba'...
remote: Enumerating objects: 425, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 425 (delta 13), reused 9 (delta 9), pack-reused 404 (from 1)
Receiving objects: 100% (425/425), 868.73 KiB | 2.91 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/PointCloudMamba
✅ 클론 완료


## Step 2. PCM 자체 causal-conv1d / mamba 설치
PCM repo 안에 포함된 소스를 직접 설치 (시스템 버전과의 API 충돌 방지)

In [ ]:
import os
os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.9'  # L4 = Ada Lovelace = sm_89

# PCM 자체 causal-conv1d 설치 (v1.0.0)
%cd /content/PointCloudMamba/openpoints/models/PCM/causal-conv1d
!pip install -e . --no-build-isolation 2>&1 | tail -3

# PCM 자체 mamba 설치 (v1.0.1)
%cd /content/PointCloudMamba/openpoints/models/PCM/mamba
!pip install -e . --no-build-isolation 2>&1 | tail -3

%cd /content/PointCloudMamba
print('✅ causal-conv1d 1.0.0 + mamba-ssm 1.0.1 설치 완료')

/content/PointCloudMamba/openpoints/models/PCM/causal-conv1d
  Running setup.py develop for causal_conv1d
/content/PointCloudMamba/openpoints/models/PCM/mamba
  Running setup.py develop for mamba_ssm
/content/PointCloudMamba
✅ causal-conv1d 1.0.0 + mamba-ssm 1.0.1 설치 완료


## Step 3. transformers 버전 고정
mamba-ssm 1.0.1은 transformers 4.38.2가 필요

In [ ]:
!pip install transformers==4.38.2 -q
print('✅ transformers 4.38.2 설치 완료')
# sentence-transformers 충돌 경고는 무시해도 됨

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.
✅ transformers 4.38.2 설치 완료


## Step 4. mamba / causal-conv1d import 검증

In [ ]:
import sys
sys.path.insert(0, '/content/PointCloudMamba/openpoints/models/PCM/causal-conv1d')
sys.path.insert(0, '/content/PointCloudMamba/openpoints/models/PCM/mamba')

# transformers 캐시 제거 (버전 변경 반영)
for key in list(sys.modules.keys()):
    if 'transformers' in key or 'mamba' in key:
        del sys.modules[key]

import causal_conv1d
import mamba_ssm
print('causal-conv1d:', causal_conv1d.__version__)  # 1.0.0
print('mamba-ssm:', mamba_ssm.__version__)           # 1.0.1
print('✅ import 성공')

/content/PointCloudMamba/openpoints/models/PCM/mamba/mamba_ssm/ops/selective_scan_interface.py:158: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/content/PointCloudMamba/openpoints/models/PCM/mamba/mamba_ssm/ops/selective_scan_interface.py:227: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/content/PointCloudMamba/openpoints/models/PCM/mamba/mamba_ssm/ops/selective_scan_interface.py:295: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/content/PointCloudMamba/openpoints/models/PCM/mamba/mamba_ssm/ops/selective_scan_interface.py:368: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  

causal-conv1d: 1.0.0
mamba-ssm: 1.0.1
✅ import 성공


## Step 5. pointnet2_batch_cuda 빌드 및 로드
cpp 모듈 빌드 후 importlib.util로 직접 로드 (egg stub 우회)

In [ ]:
import os, sys, ctypes, torch, importlib.util

os.environ['TORCH_CUDA_ARCH_LIST'] = '8.9'
os.environ['CUDA_HOME'] = '/usr/local/cuda'

# torch lib 경로 등록 (libc10.so 필요)
torch_lib = os.path.join(os.path.dirname(torch.__file__), 'lib')
os.environ['LD_LIBRARY_PATH'] = torch_lib + ':' + os.environ.get('LD_LIBRARY_PATH', '')
ctypes.CDLL(os.path.join(torch_lib, 'libc10.so'))
ctypes.CDLL(os.path.join(torch_lib, 'libtorch.so'))
ctypes.CDLL(os.path.join(torch_lib, 'libc10_cuda.so'))

# pointnet2_batch 빌드
%cd /content/PointCloudMamba/openpoints/cpp/pointnet2_batch
!python setup.py install 2>&1 | tail -3
%cd /content/PointCloudMamba

# importlib.util로 .so 직접 로드 (egg stub 우회)
so_path = '/usr/local/lib/python3.12/dist-packages/pointnet2_cuda-0.0.0-py3.12-linux-x86_64.egg/pointnet2_batch_cuda.cpython-312-x86_64-linux-gnu.so'
spec = importlib.util.spec_from_file_location('pointnet2_batch_cuda', so_path)
module = importlib.util.module_from_spec(spec)
sys.modules['pointnet2_batch_cuda'] = module
spec.loader.exec_module(module)

import pointnet2_batch_cuda
print('✅ pointnet2_batch_cuda ready')

/content/PointCloudMamba/openpoints/cpp/pointnet2_batch
Installed /usr/local/lib/python3.12/dist-packages/pointnet2_cuda-0.0.0-py3.12-linux-x86_64.egg
Processing dependencies for pointnet2-cuda==0.0.0
Finished processing dependencies for pointnet2-cuda==0.0.0
/content/PointCloudMamba
✅ pointnet2_batch_cuda ready


## Step 6. 추가 패키지 설치 + 경로 등록 + 모델 로드

In [ ]:
!pip install multimethod addict shortuuid -q

import sys
sys.path.insert(0, '/content/PointCloudMamba/openpoints/models/PCM/causal-conv1d')
sys.path.insert(0, '/content/PointCloudMamba/openpoints/models/PCM/mamba')
sys.path.insert(0, '/content/PointCloudMamba')

import torch
from openpoints.models import build_model_from_cfg
from openpoints.utils import EasyConfig

cfg = EasyConfig()
cfg.load('/content/PointCloudMamba/cfgs/scanobjectnn/PCM-Tiny.yaml', recursive=True)
model = build_model_from_cfg(cfg.model).cuda()

total_params = sum(p.numel() for p in model.parameters())
print(f'✅ PCM-Tiny 로드 성공: {total_params/1e6:.2f}M')
print(f'   GPU 여유: {torch.cuda.mem_get_info()[0]/1e9:.2f}GB')

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/pointnet2_cuda-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/PointCloudMamba/openpoints/models/layers/group.py:79: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
/content/PointCloudMamba/openpoints/models/layers/upsampling.py:46: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
/content/PointCloudMamba/openpoints/models/classification/cls_base.py:99: SyntaxWarning: invalid escape sequence '\e'
  $\eg$ cls_feat='max,avg' means use the concatenateion of maxpooled and avgpooled fea

obj_cfg:  NAME: BaseCls
encoder_args:
  NAME: PointMambaEncoder
  in_channels: 4
  embed_dim: 192
  groups: 1
  res_expansion: 1.0
  activation: relu
  bias: False
  use_xyz: True
  normalize: anchor
  dim_expansion: [1, 1, 2, 1]
  pre_blocks: [1, 1, 1, 1]
  mamba_blocks: [1, 1, 2, 2]
  pos_blocks: [0, 0, 0, 0]
  k_neighbors: [12, 12, 12, 12]
  reducers: [1, 2, 2, 2]
  rms_norm: True
  residual_in_fp32: True
  fused_add_norm: True
  bimamba_type: v2
  drop_path_rate: 0.1
  mamba_pos: True
  mamba_layers_orders: ['xyz', 'xzy', 'yxz', 'yzx', 'zxy', 'zyx']
  use_order_prompt: True
  prompt_num_per_order: 6
cls_args:
  NAME: ClsHead
  num_classes: 15
  mlps: [512, 256]
  norm_args:
    norm: bn1d
obj_cfg:  NAME: PointMambaEncoder
in_channels: 4
embed_dim: 192
groups: 1
res_expansion: 1.0
activation: relu
bias: False
use_xyz: True
normalize: anchor
dim_expansion: [1, 1, 2, 1]
pre_blocks: [1, 1, 1, 1]
mamba_blocks: [1, 1, 2, 2]
pos_blocks: [0, 0, 0, 0]
k_neighbors: [12, 12, 12, 12]
reducers:

## Step 7-1. Drive Mount

In [ ]:
import os
ckpt_dir = '/content/checkpoints/pruning'
if os.path.exists(ckpt_dir):
    print("저장된 체크포인트:", os.listdir(ckpt_dir))
else:
    print("체크포인트 폴더 없음")

체크포인트 폴더 없음


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/PCM_pruning_ckpt_v2'  # v2: batch32 실험용 새 폴더
os.makedirs(save_dir, exist_ok=True)
print(f"✅ 저장 위치: {save_dir}")

Mounted at /content/drive
✅ 저장 위치: /content/drive/MyDrive/PCM_pruning_ckpt_v2


## Step 7-2. ScanObjectNN 데이터 다운로드
**처음 한 번만 실행** (런타임 재시작해도 /content/data는 유지됨)

In [ ]:
import os

# 경로 설정
drive_cache = '/content/drive/MyDrive/ScanObjNN_minimal'  # Drive 백업 위치
local_dir   = '/content/data/ScanObjNN/h5_files/main_split'  # 로컬 작업 위치
train_file  = 'training_objectdataset_augmentedrot_scale75.h5'
test_file   = 'test_objectdataset_augmentedrot_scale75.h5'

os.makedirs(local_dir, exist_ok=True)

# ===== 경우 1: 로컬에 이미 있음 (같은 세션 재실행) =====
if os.path.exists(f'{local_dir}/{train_file}') and os.path.exists(f'{local_dir}/{test_file}'):
    print("✅ 로컬에 이미 존재 — 생략")

# ===== 경우 2: Drive 캐시에 있음 (런타임 재시작 후) =====
elif os.path.exists(f'{drive_cache}/{train_file}') and os.path.exists(f'{drive_cache}/{test_file}'):
    print("📁 Drive 캐시에서 복원 중...")
    !cp {drive_cache}/{train_file} {local_dir}/
    !cp {drive_cache}/{test_file} {local_dir}/
    print("✅ Drive에서 복원 완료 (몇 초)")

# ===== 경우 3: 어디에도 없음 (최초 1회) =====
else:
    print("⬇️  최초 다운로드 중... (HKUST 서버, 시간 걸릴 수 있음)")
    os.makedirs('/content/data/ScanObjNN', exist_ok=True)
    %cd /content/data/ScanObjNN
    !wget -q http://hkust-vgd.ust.hk/scanobjectnn/h5_files.zip
    !unzip -q h5_files.zip
    %cd /content/PointCloudMamba

    # 다운로드 후 Drive에 백업 (다음부터 빠르게)
    print("💾 Drive에 백업 중 (PB-T50-RS만)...")
    os.makedirs(drive_cache, exist_ok=True)
    !cp {local_dir}/{train_file} {drive_cache}/
    !cp {local_dir}/{test_file} {drive_cache}/
    print("✅ 다운로드 + Drive 백업 완료")

# 확인
!ls -lh {local_dir}/ | grep scale75
print("\n✅ Step 7 완료")

📁 Drive 캐시에서 복원 중...
✅ Drive에서 복원 완료 (몇 초)
-rw------- 1 root root  61M Jun 15 14:54 test_objectdataset_augmentedrot_scale75.h5
-rw------- 1 root root 242M Jun 15 14:54 training_objectdataset_augmentedrot_scale75.h5

✅ Step 7 완료


## Step 8. Dataset / DataLoader / 학습 함수 정의

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import h5py
from torch.utils.data import Dataset, DataLoader

class ScanObjectNNDataset(Dataset):
    def __init__(self, h5_path, num_points=1024, train=True):
        with h5py.File(h5_path, 'r') as f:
            self.data   = f['data'][:].astype(np.float32)
            self.labels = f['label'][:].astype(np.int64)
        self.num_points = num_points
        self.train = train

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        pts = self.data[idx]
        choice = np.random.choice(len(pts), self.num_points, replace=False)
        pts = pts[choice]
        if self.train:
            pts = pts * np.random.uniform(0.8, 1.2)
            pts += np.random.uniform(-0.2, 0.2, size=(1, 3))
        return torch.from_numpy(pts), torch.tensor(self.labels[idx])


train_path = '/content/data/ScanObjNN/h5_files/main_split/training_objectdataset_augmentedrot_scale75.h5'
test_path  = '/content/data/ScanObjNN/h5_files/main_split/test_objectdataset_augmentedrot_scale75.h5'

train_dataset = ScanObjectNNDataset(train_path, num_points=1024, train=True)
test_dataset  = ScanObjectNNDataset(test_path,  num_points=1024, train=False)

# test_loader: 평가용 (batch size는 결과에 무관)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f'✅ Train: {len(train_dataset)}개, Test: {len(test_dataset)}개')


def prepare_input(pts, labels):
    pts    = pts.cuda()
    labels = labels.cuda()
    B, N, _ = pts.shape
    ones = torch.ones(B, N, 1, device=pts.device)
    x = torch.cat([pts, ones], dim=-1).permute(0, 2, 1)
    return {'pos': pts, 'x': x, 'y': labels}

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for pts, labels in loader:
        data = prepare_input(pts, labels)
        optimizer.zero_grad()
        out  = model(data)
        loss = criterion(out, data['y'])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        total_loss += loss.item()
        correct    += (out.argmax(dim=1) == data['y']).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total * 100

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for pts, labels in loader:
            data = prepare_input(pts, labels)
            out  = model(data)
            correct += (out.argmax(dim=1) == data['y']).sum().item()
            total   += labels.size(0)
    return correct / total * 100

print('✅ 함수 정의 완료')

✅ Train: 11416개, Test: 2882개
✅ 함수 정의 완료


##10. Structured Pruning

In [ ]:
import torch, gc, os, json
import torch.nn as nn
from torch.utils.data import random_split, DataLoader
from openpoints.models import build_model_from_cfg
from openpoints.utils import EasyConfig

# Val split (seed 42 고정)
SEED = 42
g = torch.Generator().manual_seed(SEED)
val_size   = int(len(train_dataset) * 0.1)
train_size = len(train_dataset) - val_size
train_sub, val_sub = random_split(train_dataset, [train_size, val_size], generator=g)

BATCH        = 32
NUM_EPOCHS   = 250
PATIENCE     = 30
WARMUP_EPOCHS = 5

train_loader_v2 = DataLoader(train_sub, batch_size=BATCH, shuffle=True,
                             num_workers=2, drop_last=True)
val_loader_v2   = DataLoader(val_sub,   batch_size=BATCH, shuffle=False, num_workers=2)

print(f"✅ Train: {train_size} / Val: {val_size} / Test: {len(test_dataset)}")
print(f"   batch={BATCH}, lr=1e-4, warmup={WARMUP_EPOCHS}ep, max={NUM_EPOCHS}ep")

save_dir = '/content/drive/MyDrive/PCM_pruning_ckpt_v2'
os.makedirs(save_dir, exist_ok=True)

def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

def train_model(config, idx):
    ckpt_path  = f"{save_dir}/model_{idx}.pth"
    best_path  = f"{save_dir}/model_{idx}_best.pth"
    state_path = f"{save_dir}/model_{idx}_state.json"

    # 완료된 모델 스킵
    if os.path.exists(state_path):
        with open(state_path) as f:
            st = json.load(f)
        if st.get('done', False):
            print(f"⏭️  {config['label']} 완료 (Test {st['test_acc']:.2f}%) — 스킵")
            return st

    # 모델 빌드
    pcfg = EasyConfig()
    pcfg.load('/content/PointCloudMamba/cfgs/scanobjectnn/PCM-Tiny.yaml', recursive=True)
    pcfg.model.encoder_args.mamba_blocks = config['mamba_blocks']
    pcfg.model.encoder_args.mamba_layers_orders = config['orders']

    gc.collect(); torch.cuda.empty_cache()
    model = build_model_from_cfg(pcfg.model).cuda()
    params = count_params(model)

    # Optimizer + Scheduler (PCM 논문 원본)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS)
    warmup_sched = torch.optim.lr_scheduler.LinearLR(
        opt, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS
    )
    crit = torch.nn.CrossEntropyLoss(label_smoothing=0.2)

    start_epoch, best_val, best_epoch, no_improve = 1, 0.0, 0, 0

    # 이어하기
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path)
        model.load_state_dict(ck['model'])
        opt.load_state_dict(ck['opt'])
        cosine_sched.load_state_dict(ck['cosine_sched'])
        warmup_sched.load_state_dict(ck['warmup_sched'])
        start_epoch = ck['epoch'] + 1
        best_val    = ck['best_val']
        best_epoch  = ck['best_epoch']
        no_improve  = ck['no_improve']
        print(f"🔄 {config['label']} Epoch {start_epoch}부터 이어서 (best_val {best_val:.2f}%)")

    print(f"\n{'='*60}")
    print(f"{config['label']} ({params:.2f}M) | lr=1e-4, warmup={WARMUP_EPOCHS}ep, batch={BATCH}")
    print('='*60)

    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        # 학습
        train_one_epoch(model, train_loader_v2, opt, crit)
        val_acc = evaluate(model, val_loader_v2)

        # Warmup → Cosine 전환
        if epoch <= WARMUP_EPOCHS:
            warmup_sched.step()
            lr_now = opt.param_groups[0]['lr']
        else:
            cosine_sched.step()
            lr_now = opt.param_groups[0]['lr']

        # Best 갱신
        if val_acc > best_val:
            best_val, best_epoch, no_improve = val_acc, epoch, 0
            torch.save(model.state_dict(), best_path)
        else:
            no_improve += 1

        # 매 epoch Drive 저장 (이어하기용)
        torch.save({
            'model': model.state_dict(),
            'opt': opt.state_dict(),
            'cosine_sched': cosine_sched.state_dict(),
            'warmup_sched': warmup_sched.state_dict(),
            'epoch': epoch,
            'best_val': best_val,
            'best_epoch': best_epoch,
            'no_improve': no_improve,
        }, ckpt_path)

        if epoch % 10 == 0 or epoch <= 5:
            print(f"  Epoch {epoch:3d} | Val: {val_acc:.2f}% | best: {best_val:.2f}% (ep {best_epoch}) | lr: {lr_now:.2e} | no_improve: {no_improve}")

        if no_improve >= PATIENCE:
            print(f"  ⏹️  Early stop @ epoch {epoch}")
            break

    # val-best로 test 1회 평가
    model.load_state_dict(torch.load(best_path))
    test_acc = evaluate(model, test_loader)

    rec = {'label': config['label'], 'params': params,
           'test_acc': test_acc, 'best_val': best_val,
           'best_epoch': best_epoch, 'done': True}
    with open(state_path, 'w') as f:
        json.dump(rec, f)

    print(f"✅ {config['label']} 완료 → Test OA: {test_acc:.2f}% (val-best ep {best_epoch})")
    return rec

print("✅ train_model 함수 정의 완료")

✅ Train: 10275 / Val: 1141 / Test: 2882
   batch=32, lr=1e-4, warmup=5ep, max=250ep
✅ train_model 함수 정의 완료


In [ ]:
config_0 = {
    'mamba_blocks': [1,1,2,2],
    'orders': ['xyz','xzy','yxz','yzx','zxy','zyx'],
    'label': 'PCM-Tiny 원본'
}
result_0 = train_model(config_0, idx=0)

obj_cfg:  NAME: BaseCls
encoder_args:
  NAME: PointMambaEncoder
  in_channels: 4
  embed_dim: 192
  groups: 1
  res_expansion: 1.0
  activation: relu
  bias: False
  use_xyz: True
  normalize: anchor
  dim_expansion: [1, 1, 2, 1]
  pre_blocks: [1, 1, 1, 1]
  mamba_blocks: [1, 1, 2, 2]
  pos_blocks: [0, 0, 0, 0]
  k_neighbors: [12, 12, 12, 12]
  reducers: [1, 2, 2, 2]
  rms_norm: True
  residual_in_fp32: True
  fused_add_norm: True
  bimamba_type: v2
  drop_path_rate: 0.1
  mamba_pos: True
  mamba_layers_orders: ['xyz', 'xzy', 'yxz', 'yzx', 'zxy', 'zyx']
  use_order_prompt: True
  prompt_num_per_order: 6
cls_args:
  NAME: ClsHead
  num_classes: 15
  mlps: [512, 256]
  norm_args:
    norm: bn1d
obj_cfg:  NAME: PointMambaEncoder
in_channels: 4
embed_dim: 192
groups: 1
res_expansion: 1.0
activation: relu
bias: False
use_xyz: True
normalize: anchor
dim_expansion: [1, 1, 2, 1]
pre_blocks: [1, 1, 1, 1]
mamba_blocks: [1, 1, 2, 2]
pos_blocks: [0, 0, 0, 0]
k_neighbors: [12, 12, 12, 12]
reducers:

/content/PointCloudMamba/openpoints/models/layers/subsample.py:93: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  output = torch.cuda.IntTensor(B, npoint)


  Epoch 130 | Val: 96.06% | best: 97.02% (ep 129) | lr: 5.00e-05 | no_improve: 1
  Epoch 140 | Val: 96.14% | best: 97.02% (ep 129) | lr: 4.37e-05 | no_improve: 11
  Epoch 150 | Val: 96.93% | best: 97.28% (ep 141) | lr: 3.76e-05 | no_improve: 9
  Epoch 160 | Val: 96.84% | best: 97.72% (ep 159) | lr: 3.16e-05 | no_improve: 1
  Epoch 170 | Val: 96.49% | best: 97.72% (ep 159) | lr: 2.59e-05 | no_improve: 11
  Epoch 180 | Val: 97.11% | best: 97.72% (ep 159) | lr: 2.06e-05 | no_improve: 21
  ⏹️  Early stop @ epoch 189
✅ PCM-Tiny 원본 완료 → Test OA: 83.83% (val-best ep 159)


In [ ]:
config_1 = {
    'mamba_blocks': [1,1,1,1],
    'orders': ['xyz','xzy','yxz','yzx'],
    'label': '중간 (4.85M)'
}
result_1 = train_model(config_1, idx=1)

obj_cfg:  NAME: BaseCls
encoder_args:
  NAME: PointMambaEncoder
  in_channels: 4
  embed_dim: 192
  groups: 1
  res_expansion: 1.0
  activation: relu
  bias: False
  use_xyz: True
  normalize: anchor
  dim_expansion: [1, 1, 2, 1]
  pre_blocks: [1, 1, 1, 1]
  mamba_blocks: [1, 1, 1, 1]
  pos_blocks: [0, 0, 0, 0]
  k_neighbors: [12, 12, 12, 12]
  reducers: [1, 2, 2, 2]
  rms_norm: True
  residual_in_fp32: True
  fused_add_norm: True
  bimamba_type: v2
  drop_path_rate: 0.1
  mamba_pos: True
  mamba_layers_orders: ['xyz', 'xzy', 'yxz', 'yzx']
  use_order_prompt: True
  prompt_num_per_order: 6
cls_args:
  NAME: ClsHead
  num_classes: 15
  mlps: [512, 256]
  norm_args:
    norm: bn1d
obj_cfg:  NAME: PointMambaEncoder
in_channels: 4
embed_dim: 192
groups: 1
res_expansion: 1.0
activation: relu
bias: False
use_xyz: True
normalize: anchor
dim_expansion: [1, 1, 2, 1]
pre_blocks: [1, 1, 1, 1]
mamba_blocks: [1, 1, 1, 1]
pos_blocks: [0, 0, 0, 0]
k_neighbors: [12, 12, 12, 12]
reducers: [1, 2, 2, 2]


In [ ]:
config_2 = {
    'mamba_blocks': [1,0,0,0],
    'orders': ['xyz'],
    'label': '최소 (2.48M)'
}
result_2 = train_model(config_2, idx=2)

obj_cfg:  NAME: BaseCls
encoder_args:
  NAME: PointMambaEncoder
  in_channels: 4
  embed_dim: 192
  groups: 1
  res_expansion: 1.0
  activation: relu
  bias: False
  use_xyz: True
  normalize: anchor
  dim_expansion: [1, 1, 2, 1]
  pre_blocks: [1, 1, 1, 1]
  mamba_blocks: [1, 0, 0, 0]
  pos_blocks: [0, 0, 0, 0]
  k_neighbors: [12, 12, 12, 12]
  reducers: [1, 2, 2, 2]
  rms_norm: True
  residual_in_fp32: True
  fused_add_norm: True
  bimamba_type: v2
  drop_path_rate: 0.1
  mamba_pos: True
  mamba_layers_orders: ['xyz']
  use_order_prompt: True
  prompt_num_per_order: 6
cls_args:
  NAME: ClsHead
  num_classes: 15
  mlps: [512, 256]
  norm_args:
    norm: bn1d
obj_cfg:  NAME: PointMambaEncoder
in_channels: 4
embed_dim: 192
groups: 1
res_expansion: 1.0
activation: relu
bias: False
use_xyz: True
normalize: anchor
dim_expansion: [1, 1, 2, 1]
pre_blocks: [1, 1, 1, 1]
mamba_blocks: [1, 0, 0, 0]
pos_blocks: [0, 0, 0, 0]
k_neighbors: [12, 12, 12, 12]
reducers: [1, 2, 2, 2]
rms_norm: True
residu

In [ ]:
import os, json

print(f"{'Model':<20} | {'Params':>7} | {'Test OA':>8} | best ep")
print('-'*55)
for i in [0, 1, 2]:
    sp = f"{save_dir}/model_{i}_state.json"
    if os.path.exists(sp):
        with open(sp) as f:
            st = json.load(f)
        print(f"{st['label']:<20} | {st['params']:>6.2f}M | {st['test_acc']:>7.2f}% | {st['best_epoch']}")

Model                |  Params |  Test OA | best ep
-------------------------------------------------------
PCM-Tiny 원본          |   6.94M |   83.83% | 159
중간 (4.85M)           |   4.85M |   83.59% | 157
최소 (2.48M)           |   2.48M |   82.23% | 223
